# Setup environment

In [1]:
%pip install -q "crewai[tools, agentops]==0.114.0"
%pip install -q python-dotenv tavily-python

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
from tavily import TavilyClient
import agentops
import os

load_dotenv()
agentsops_api_key = os.getenv("AGENTSOPS_API_KEY")
llm_api_key = os.getenv("GEMINI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

agentops.init(
    api_key = agentsops_api_key,
    skip_auto_end_session = True,
    default_tags = ['crewai']
)

basic_llm = LLM(
    model = "gemini-2.0-flash",
    api_key = llm_api_key,
    temperature = 0
)

search_client = TavilyClient(api_key = tavily_api_key)


In [3]:
# test tavily search
search_client.search("What is the capital of France?")

{'query': 'What is the capital of France?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_France',
   'title': 'List of capitals of France',
   'content': 'This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944.\n\n## Chronology\n\n[edit] [...] Tours (10–13 June 1940), the city served as the temporary capital of France during World War II after the government fled Paris due to the German advance.\n Bordeaux (June 1940), the government was relocated from Paris to Tours then Bordeaux very briefly during World War II, when it became apparent that Paris would soon fall into German hands. [...] Algiers (1943–1944), the city was made the seat of Free France, to be closer to the war in Europe.\n Paris (1945–present day).',
   'score': 0.8875269,
   'raw_content': None,
   'id': '9df6ed-00'},
  {'url': 'https://home.adelphi.edu/~ca19535/page%204

In [4]:
output_dir = "./ai-agent-output"
os.makedirs(output_dir, exist_ok=True)

# Setup Agents

## Get Queries Agent

In [5]:
no_keywords = 10

# example output from the agent
{
    "queries" : [
        "keyword1", "keyword2", "keyword3"
    ]
}

{'queries': ['keyword1', 'keyword2', 'keyword3']}

In [6]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(...,   # for required field
                               description = "A list of suggested search queries to be passed to the search engine.",
                               min_length = 1,
                               max_length = no_keywords
                        )

search_queries_recommendation_agent = Agent(
    role = "Search Queries Recommendation Agent",
    goal = "\n".join([
        "To provide a list of suggested search queries to be passed to the search engine.",
        "The queries must be varied and looking for specific items."
    ]),
    backstory = "The agent is designed to help in looking for products by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
    llm = basic_llm,
    verbose = True
)

search_queries_recommendation_task = Task(
    description = "\n".join([
        "Rankyx is looking to buy {product_name} at the best prices (value for a price strategy)",
        "The company target any of these websites to buy from: {websites_list}",
        "The company wants to reach all available proucts on the internet to be compared later in another stage.",
        "The stores must sell the product in {country_name}",
        "Generate at maximum {no_keywords} queries.",
        "The search keywords must be in {language} language.",
        "Search keywords must contains specific brands, types or technologies. Avoid general keywords.",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]),
    expected_output = "A JSON object containing a list of suggested search queries.",
    output_json = SuggestedSearchQueries,
    output_file = os.path.join(output_dir, "step_1_suggested_search_queries.json"),
    agent = search_queries_recommendation_agent
)

## Search Agent

In [7]:
{
    "results" : [
        {
            "title" : "",
            "url" : "",
            "content" : "",
            "score" : 0.50,
            "search_query" : ""
        },
        {
            "title" : "",
            "url" : "",
            "content" : "",
            "score" : 0.60,
            "search_query" : ""
        }
    ]
}

{'results': [{'title': '',
   'url': '',
   'content': '',
   'score': 0.5,
   'search_query': ''},
  {'title': '', 'url': '', 'content': '', 'score': 0.6, 'search_query': ''}]}

In [8]:
class SignleSearchResult(BaseModel):
    title: str
    url: str = Field(..., title = "The page url")
    content: str
    score: float
    search_query: str

class AllSearchResults(BaseModel):
    results: List[SignleSearchResult]

@tool
def search_engine_tool(query: str) -> str:
    """Useful for search-based queries. Use this to find current information about any query related pages using a search engine."""
    return search_client.search(query)
    

search_engine_agent = Agent(
    role = "Search Engine Agent",
    goal = "To search for products based on the suggested search query",
    backstory = "The agent is designed to help in looking for products by searching for products based on the suggested search queries.",
    llm = basic_llm,
    verbose = True,
    tools = [search_engine_tool]
)

search_engine_task = Task(
    description="\n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any susbicious links or not an ecommerce single product website link.",
        "Ignore any search results with confidence score less than ({score_threshold}) .",
        "The search results will be used to compare prices of products from different websites."
    ]),
    expected_output = "A JSON object containing the search results.",
    output_json = AllSearchResults,
    output_file = os.path.join(output_dir, "step_2_search_results.json"),
    agent = search_engine_agent
)

# Run Crew AI

In [9]:
rankyx_crew = Crew(
    agents = [
        search_queries_recommendation_agent,
        search_engine_agent
        ],
    tasks = [
        search_queries_recommendation_task,
        search_engine_task
    ],
    process = Process.sequential,
)

In [10]:
crew_results = rankyx_crew.kickoff(
    inputs = {
        "product_name": "coffee machine for the office",
        "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
        "country_name": "Egypt",
        "no_keywords": 10,
        "language": "English",
        "score_threshold": 0.10
    }
)

# Agent: Search Queries Recommendation Agent
## Task: Rankyx is looking to buy coffee machine for the office at the best prices (value for a price strategy)
The company target any of these websites to buy from: ['www.amazon.eg', 'www.jumia.com.eg', 'www.noon.com/egypt-en']
The company wants to reach all available proucts on the internet to be compared later in another stage.
The stores must sell the product in Egypt
Generate at maximum 10 queries.
The search keywords must be in English language.
Search keywords must contains specific brands, types or technologies. Avoid general keywords.
The search query must reach an ecommerce webpage for product, and not a blog or listing page.


# Agent: Search Queries Recommendation Agent
## Final Answer: 
{
  "queries": [
    "site:amazon.eg DeLonghi Dedica EC685 15 bar espresso machine",
    "site:noon.com/egypt-en Black Decker 12-cup drip coffee maker DCM85",
    "site:jumia.com.eg Sonai espresso coffee maker milk frother 15 bar",
    "site:amaz

🖇 AgentOps: ToolEvent() is deprecated and will be removed in v4 in the future. Automatically tracked in v4.




# Agent: Search Engine Agent
## Thought: Action: search_engine_tool
## Using tool: search_engine_tool
## Tool Input: 
"{\"query\": \"site:amazon.eg DeLonghi Dedica EC685 15 bar espresso machine\"}"
## Tool Output: 
{'query': 'DeLonghi Dedica EC685 15 bar espresso machine', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.amazon.eg/-/en/DeLonghi-Dedica-Adjustable-Black-Espresso/dp/B0BFM9FP4N', 'title': 'DeLonghi Dedica EC685. BK Adjustable Black Espresso Maker: Buy Online at Best Price in Egypt - Souq is now Amazon.eg', 'content': "|  |  |\n --- |\n| Other Special Features of the Product | Milk Frother, Removable Tank |\n| Coffee Maker Type | Espresso Machine |\n| Specific Uses For Product | Cappuccino |\n| Recommended Uses For Product | Fishing |\n| Operation Mode | Semi-Automatic |\n| Wattage | 1300 watts |\n| Voltage | 220 Volts |\n| Human Interface Input | Buttons |\n| Output Pressure | 15 Bars |\n| Coffee Input Type | ground\\_coffee\\_o

ERROR:root:LiteLLM call failed: litellm.BadRequestError: VertexAIException BadRequestError - {
  "error": {
    "code": 400,
    "message": "Requests ending with a model turn are not supported.",
    "status": "INVALID_ARGUMENT"
  }
}

🖇 AgentOps: Error in span gemini/gemini-3.6-flash.llm: litellm.BadRequestError: VertexAIException BadRequestError - {
  "error": {
    "code": 400,
    "message": "Requests ending with a model turn are not supported.",
    "status": "INVALID_ARGUMENT"
  }
}

🖇 AgentOps: Error in span Search Engine Agent.agent: litellm.BadRequestError: VertexAIException BadRequestError - {
  "error": {
    "code": 400,
    "message": "Requests ending with a model turn are not supported.",
    "status": "INVALID_ARGUMENT"
  }
}

🖇 AgentOps: Error in span The task is to search for products based on the suggested search queries.
You have to collect results from multiple search queries.
Ignore any susbicious links or not an ecommerce single product website link.
Ignore any se



LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

 Error during LLM call: litellm.BadRequestError: VertexAIException BadRequestError - {
  "error": {
    "code": 400,
    "message": "Requests ending with a model turn are not supported.",
    "status": "INVALID_ARGUMENT"
  }
}

 An unknown error occurred. Please check the details below.
 Error details: litellm.BadRequestError: VertexAIException BadRequestError - {
  "error": {
    "code": 400,
    "message": "Requests ending with a model turn are not supported.",
    "status": "INVALID_ARGUMENT"
  }
}



🖇 AgentOps: [agentops.InternalSpanProcessor] Error uploading logfile: Upload failed: 401


BadRequestError: litellm.BadRequestError: VertexAIException BadRequestError - {
  "error": {
    "code": 400,
    "message": "Requests ending with a model turn are not supported.",
    "status": "INVALID_ARGUMENT"
  }
}
